# 03 Silver - Telecommunications

**Audience:** participants learning the AIDP medallion pattern with PySpark.

**Prerequisites:** use the lab's shared compute and run the previous notebook first.

**Learning goal:** Casts, normalizes, deduplicates, validates relationships, and quarantines invalid rows.

## Outline

1. Inspect the participant-scoped inputs.
2. Transform and persist this medallion layer.
3. Register external tables when this layer owns them.
4. Verify the row counts printed by the final statements.


In [ ]:
import re
import oidlUtils

def required_parameter(name):
    value = oidlUtils.parameters.getParameter(name)
    if value is None or not str(value).strip():
        raise ValueError(f"Missing AIDP job parameter: {name}")
    return str(value).strip()

participant_key = required_parameter("participant_key")
lab_id = required_parameter("lab_id")
workspace_root = required_parameter("workspace_root")
bucket_name = required_parameter("bucket_name")
objectstorage_namespace = required_parameter("objectstorage_namespace")

if re.fullmatch(r"u_[0-9a-f]{16}", participant_key) is None:
    raise ValueError("Invalid participant_key")
if lab_id != 'telecommunications':
    raise ValueError("This notebook belongs to a different lab")
if not workspace_root.startswith("/Workspace/medallon/"):
    raise ValueError("Invalid workspace_root")

from functools import reduce
from pyspark.sql import Window, functions as F

industry = 'telecommunications'
specs = {'plans': {'filename': 'plans.csv', 'primary_key': ['plan_id'], 'foreign_keys': [], 'columns': [{'name': 'participant_key', 'type': 'STRING', 'required': True}, {'name': 'source_row_id', 'type': 'STRING', 'required': True}, {'name': 'plan_id', 'type': 'STRING', 'required': True}, {'name': 'plan_type', 'type': 'STRING', 'required': True}, {'name': 'monthly_fee', 'type': 'DOUBLE', 'required': True}, {'name': 'included_data_mb', 'type': 'BIGINT', 'required': True}, {'name': 'included_voice_minutes', 'type': 'BIGINT', 'required': True}, {'name': 'overage_rate', 'type': 'DOUBLE', 'required': True}, {'name': 'status', 'type': 'STRING', 'required': True}, {'name': 'updated_at', 'type': 'TIMESTAMP', 'required': True}]}, 'network_sites': {'filename': 'network_sites.csv', 'primary_key': ['site_id'], 'foreign_keys': [], 'columns': [{'name': 'participant_key', 'type': 'STRING', 'required': True}, {'name': 'source_row_id', 'type': 'STRING', 'required': True}, {'name': 'site_id', 'type': 'STRING', 'required': True}, {'name': 'region', 'type': 'STRING', 'required': True}, {'name': 'technology', 'type': 'STRING', 'required': True}, {'name': 'capacity_mb_day', 'type': 'BIGINT', 'required': True}, {'name': 'commissioned_date', 'type': 'DATE', 'required': True}, {'name': 'status', 'type': 'STRING', 'required': True}, {'name': 'updated_at', 'type': 'TIMESTAMP', 'required': True}]}, 'subscribers': {'filename': 'subscribers.csv', 'primary_key': ['subscriber_id'], 'foreign_keys': [['plan_id', 'plans', 'plan_id'], ['home_site_id', 'network_sites', 'site_id']], 'columns': [{'name': 'participant_key', 'type': 'STRING', 'required': True}, {'name': 'source_row_id', 'type': 'STRING', 'required': True}, {'name': 'subscriber_id', 'type': 'STRING', 'required': True}, {'name': 'plan_id', 'type': 'STRING', 'required': True}, {'name': 'home_site_id', 'type': 'STRING', 'required': True}, {'name': 'segment', 'type': 'STRING', 'required': True}, {'name': 'region', 'type': 'STRING', 'required': True}, {'name': 'activation_date', 'type': 'DATE', 'required': True}, {'name': 'status', 'type': 'STRING', 'required': True}, {'name': 'updated_at', 'type': 'TIMESTAMP', 'required': True}]}, 'usage_events': {'filename': 'usage_events.csv', 'primary_key': ['event_id'], 'foreign_keys': [['subscriber_id', 'subscribers', 'subscriber_id'], ['site_id', 'network_sites', 'site_id']], 'columns': [{'name': 'participant_key', 'type': 'STRING', 'required': True}, {'name': 'source_row_id', 'type': 'STRING', 'required': True}, {'name': 'event_id', 'type': 'STRING', 'required': True}, {'name': 'subscriber_id', 'type': 'STRING', 'required': True}, {'name': 'site_id', 'type': 'STRING', 'required': True}, {'name': 'event_time', 'type': 'TIMESTAMP', 'required': True}, {'name': 'usage_type', 'type': 'STRING', 'required': True}, {'name': 'usage_value', 'type': 'DOUBLE', 'required': True}, {'name': 'usage_unit', 'type': 'STRING', 'required': True}, {'name': 'charge_amount', 'type': 'DOUBLE', 'required': True}, {'name': 'updated_at', 'type': 'TIMESTAMP', 'required': True}]}}
sources = {"network_sites": f"oci://{bucket_name}@{objectstorage_namespace}/02_bronze/users/{participant_key}/telecommunications/network_sites/", "plans": f"oci://{bucket_name}@{objectstorage_namespace}/02_bronze/users/{participant_key}/telecommunications/plans/", "subscribers": f"oci://{bucket_name}@{objectstorage_namespace}/02_bronze/users/{participant_key}/telecommunications/subscribers/", "usage_events": f"oci://{bucket_name}@{objectstorage_namespace}/02_bronze/users/{participant_key}/telecommunications/usage_events/"}
destinations = {"network_sites": f"oci://{bucket_name}@{objectstorage_namespace}/03_silver/users/{participant_key}/telecommunications/network_sites/", "plans": f"oci://{bucket_name}@{objectstorage_namespace}/03_silver/users/{participant_key}/telecommunications/plans/", "subscribers": f"oci://{bucket_name}@{objectstorage_namespace}/03_silver/users/{participant_key}/telecommunications/subscribers/", "usage_events": f"oci://{bucket_name}@{objectstorage_namespace}/03_silver/users/{participant_key}/telecommunications/usage_events/"}
quality_uri = f'oci://{bucket_name}@{objectstorage_namespace}/03_silver/users/{participant_key}/telecommunications/quality_issues/'
spark_types = {"STRING": "string", "DATE": "date", "TIMESTAMP": "timestamp", "DOUBLE": "double", "BIGINT": "bigint", "BOOLEAN": "boolean"}
enum_rules = {"network_sites": {"region": ["north", "south", "central", "coastal"], "status": ["active"], "technology": ["4g", "5g"]}, "plans": {"plan_type": ["prepaid", "postpaid", "business"], "status": ["active"]}, "subscribers": {"region": ["north", "south", "central", "coastal"], "segment": ["consumer", "family", "business"], "status": ["active"]}, "usage_events": {"usage_type": ["data", "voice", "sms"], "usage_unit": ["mb", "minutes", "messages"]}}
positive_rules = {"network_sites": ["capacity_mb_day"], "usage_events": ["usage_value"]}
temporal_rules = {}

typed = {}
for dataset, spec in specs.items():
    frame = spark.read.format("delta").load(sources[dataset])
    cast_failure_columns = []
    for column in spec["columns"]:
        name, kind = column["name"], column["type"]
        raw_text = F.trim(F.col(name).cast("string"))
        if kind == "STRING":
            normalized = F.when(raw_text == "", F.lit(None)).otherwise(raw_text)
            if name not in {"participant_key", "source_row_id"} and not name.endswith("_id"):
                normalized = F.lower(normalized)
            frame = frame.withColumn(name, normalized)
        else:
            flag = f"_invalid_cast_{name}"
            frame = frame.withColumn(
                flag,
                raw_text.isNotNull() & (raw_text != "") & raw_text.cast(spark_types[kind]).isNull(),
            ).withColumn(name, raw_text.cast(spark_types[kind]))
            cast_failure_columns.append(flag)
    cast_invalid = reduce(
        lambda left, name: left | F.col(name), cast_failure_columns, F.lit(False)
    )
    frame = frame.withColumn("_cast_invalid", cast_invalid).drop(*cast_failure_columns)
    typed[dataset] = frame

quality_frames = []
accepted = {}
for dataset, spec in specs.items():
    frame = typed[dataset]
    required_checks = [
        F.col(column["name"]).isNull()
        | ((F.col(column["name"]) == "") if column["type"] == "STRING" else F.lit(False))
        for column in spec["columns"] if column["required"]
    ]
    required_invalid = reduce(lambda left, check: left | check, required_checks, F.lit(False))
    key_window = Window.partitionBy(*spec["primary_key"]).orderBy(F.col("updated_at").desc_nulls_last(), F.col("source_row_id").desc())
    frame = (frame.withColumn("_duplicate_rank", F.row_number().over(key_window))
        .withColumn("_fk_invalid", F.lit(False)))
    for local_column, reference_dataset, reference_column in spec["foreign_keys"]:
        marker = f"_ref_{dataset}_{local_column}"
        reference = accepted[reference_dataset].select(F.col(reference_column).alias(marker)).distinct()
        frame = frame.join(F.broadcast(reference), frame[local_column] == reference[marker], "left")
        frame = frame.withColumn(
            "_fk_invalid",
            F.col("_fk_invalid") | (F.col(local_column).isNotNull() & F.col(marker).isNull()),
        ).drop(marker)
    fk_invalid = F.col("_fk_invalid")
    enum_invalid = reduce(
        lambda left, item: left | (F.col(item[0]).isNotNull() & ~F.col(item[0]).isin(item[1])),
        enum_rules.get(dataset, {}).items(),
        F.lit(False),
    )
    range_invalid = reduce(
        lambda left, name: left | (F.col(name).isNotNull() & (F.col(name) <= 0)),
        positive_rules.get(dataset, []),
        F.lit(False),
    )
    temporal_invalid = reduce(
        lambda left, pair: left | (
            F.col(pair[0]).isNotNull()
            & F.col(pair[1]).isNotNull()
            & (F.col(pair[1]) <= F.col(pair[0]))
        ),
        temporal_rules.get(dataset, []),
        F.lit(False),
    )
    scope_invalid = F.col("participant_key") != F.lit(participant_key)
    duplicate_invalid = F.col("_duplicate_rank") > 1
    quality_invalid = (
        required_invalid | F.col("_cast_invalid") | fk_invalid | enum_invalid
        | range_invalid | temporal_invalid | scope_invalid | duplicate_invalid
    )
    frame = frame.withColumn("_quality_invalid", quality_invalid)
    reason = F.concat_ws(",",
        F.when(required_invalid, F.lit("required_value")),
        F.when(F.col("_cast_invalid"), F.lit("invalid_type")),
        F.when(fk_invalid, F.lit("foreign_key")),
        F.when(enum_invalid, F.lit("invalid_enum")),
        F.when(range_invalid, F.lit("invalid_range")),
        F.when(temporal_invalid, F.lit("invalid_time_order")),
        F.when(scope_invalid, F.lit("participant_scope")),
        F.when(duplicate_invalid, F.lit("duplicate_key")),
    )
    source_columns = [column["name"] for column in spec["columns"]]
    issues = (frame.filter(F.col("_quality_invalid"))
        .withColumn("industry", F.lit(industry))
        .withColumn("dataset", F.lit(dataset))
        .withColumn("record_key", F.concat_ws("|", *[F.col(name).cast("string") for name in spec["primary_key"]]))
        .withColumn("reason_codes", reason)
        .withColumn("raw_payload_json", F.to_json(F.struct(*[F.col(name) for name in source_columns])))
        .withColumn("quarantined_at", F.current_timestamp())
        .select("participant_key", "industry", "dataset", "source_row_id", "record_key", "reason_codes", "raw_payload_json", "quarantined_at"))
    quality_frames.append(issues)
    clean = frame.filter(~F.col("_quality_invalid")).select(*source_columns)
    accepted[dataset] = clean
    bronze_count = frame.count()
    clean_count = clean.count()
    assert clean_count <= bronze_count, f"Silver count increased for {dataset}"
    clean.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(destinations[dataset])
    print(f"Silver {dataset}: {clean_count} accepted rows")

quality = reduce(lambda left, right: left.unionByName(right), quality_frames)
quality.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(quality_uri)
quality_count = quality.count()
assert quality_count > 0, "The deterministic lab data must exercise the quarantine path"
print(f"Quality issues: {quality_count} rows")

spark.sql(f"""CREATE EXTERNAL TABLE IF NOT EXISTS aidp_lab.oci_silver.{participant_key}_telecommunications_plans (`participant_key` STRING, `source_row_id` STRING, `plan_id` STRING, `plan_type` STRING, `monthly_fee` DOUBLE, `included_data_mb` BIGINT, `included_voice_minutes` BIGINT, `overage_rate` DOUBLE, `status` STRING, `updated_at` TIMESTAMP) USING DELTA LOCATION 'oci://{bucket_name}@{objectstorage_namespace}/03_silver/users/{participant_key}/telecommunications/plans/'""")
spark.sql(f"""CREATE EXTERNAL TABLE IF NOT EXISTS aidp_lab.oci_silver.{participant_key}_telecommunications_network_sites (`participant_key` STRING, `source_row_id` STRING, `site_id` STRING, `region` STRING, `technology` STRING, `capacity_mb_day` BIGINT, `commissioned_date` DATE, `status` STRING, `updated_at` TIMESTAMP) USING DELTA LOCATION 'oci://{bucket_name}@{objectstorage_namespace}/03_silver/users/{participant_key}/telecommunications/network_sites/'""")
spark.sql(f"""CREATE EXTERNAL TABLE IF NOT EXISTS aidp_lab.oci_silver.{participant_key}_telecommunications_subscribers (`participant_key` STRING, `source_row_id` STRING, `subscriber_id` STRING, `plan_id` STRING, `home_site_id` STRING, `segment` STRING, `region` STRING, `activation_date` DATE, `status` STRING, `updated_at` TIMESTAMP) USING DELTA LOCATION 'oci://{bucket_name}@{objectstorage_namespace}/03_silver/users/{participant_key}/telecommunications/subscribers/'""")
spark.sql(f"""CREATE EXTERNAL TABLE IF NOT EXISTS aidp_lab.oci_silver.{participant_key}_telecommunications_usage_events (`participant_key` STRING, `source_row_id` STRING, `event_id` STRING, `subscriber_id` STRING, `site_id` STRING, `event_time` TIMESTAMP, `usage_type` STRING, `usage_value` DOUBLE, `usage_unit` STRING, `charge_amount` DOUBLE, `updated_at` TIMESTAMP) USING DELTA LOCATION 'oci://{bucket_name}@{objectstorage_namespace}/03_silver/users/{participant_key}/telecommunications/usage_events/'""")
spark.sql(f"""CREATE EXTERNAL TABLE IF NOT EXISTS aidp_lab.oci_silver.{participant_key}_telecommunications_quality_issues (`participant_key` STRING, `industry` STRING, `dataset` STRING, `source_row_id` STRING, `record_key` STRING, `reason_codes` STRING, `raw_payload_json` STRING, `quarantined_at` TIMESTAMP) USING DELTA LOCATION 'oci://{bucket_name}@{objectstorage_namespace}/03_silver/users/{participant_key}/telecommunications/quality_issues/'""")


## Expected result

Four clean Delta tables and a non-empty `quality_issues` table are registered.

**Exercise:** rerun this notebook and confirm that counts do not increase. All writes use
participant-exclusive paths and overwrite mode, so a second run is idempotent.

**Common pitfall:** do not replace the participant paths with shared locations. That would mix
different students' data. As an extension, query the registered tables with `spark.sql`.
